# 1、引入BitsAndBytesConfig类，构建一个量化配置

In [21]:
from transformers import BitsAndBytesConfig
import torch
quantization_config = BitsAndBytesConfig(
    load_in_4bit= True,
    bnb_4bit_quant_type="nf4", # 使用nf4进行4bit量化
    bnb_4bit_use_double_quant=False, # 是否使用双重量化
    bnb_4bit_compute_dtype=torch.bfloat16,  # 反量化成 torch.bf16，做运算
)

# 2、加载模型时，传入quantization_config

In [22]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B",quantization_config = quantization_config)
model

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 358.90it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear4bit(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
 

In [24]:
model.lm_head.weight.dtype

torch.bfloat16

# 3、通过peft_model，加载prepare_model_for_kbit_training

In [25]:
from peft import prepare_model_for_kbit_training

# prepare_model_for_kbit_training
# 1、将模型内部关键组件精度升高成fp32
# 2、将线性层的梯度关闭，requires_grad置为False (和get_peft_model职责类似)

model = prepare_model_for_kbit_training(model)

In [26]:
model.lm_head.weight.dtype

torch.float32

In [27]:
from peft import LoraConfig
from transformers import AutoModelForCausalLM,AutoTokenizer
from datasets import load_dataset, DatasetDict
## 1.1 数据加载
data:DatasetDict = load_dataset("json",data_files={"train":"data/psychology_data.jsonl",})

In [28]:
data = data["train"].train_test_split(test_size = 0.2)

In [29]:
from typing import Dict,List
def convert_type(examples:Dict[str, List]):
    """
    讲数据，转换成 SFTTrainer所需要的Language Modeling类型，对话格式
    """
    conversation_list:List[List[Dict]] = examples["conversation"]

    all_data_messages_list = []

    for data in conversation_list:
        human_message = data[0]["human"]
        assistant_message = data[0]["assistant"]

        message_list = [
            {"role":"user","content":human_message},
            {"role":"assistant","content":assistant_message}
        ]

        all_data_messages_list.append(message_list)

    return {"messages":all_data_messages_list}


# batched=True，传递给convert_type的是一批数据，
mapped_data = data.map(convert_type,batched=True,remove_columns=['conversation_id', 'category', 'conversation', 'dataset'])
mapped_data

Map: 100%|██████████| 15450/15450 [00:00<00:00, 78031.64 examples/s]


DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 61800
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 15450
    })
})

In [15]:
from peft import LoraConfig,get_peft_model

model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B/")
tokenizer = AutoTokenizer.from_pretrained("model/Qwen3-0.6B/")
# 3、构造LoraConfig
lora_config = LoraConfig(
    r = 16,
    lora_alpha = 16,
    # target_modules = ["q_proj","v_proj"]
    # target_modules = ["q_proj","v_proj","k_proj","o_proj","gate_proj","up_proj","down_proj"]
    target_modules = "all-linear",
    lora_dropout = 0.05,
    task_type = "CAUSAL_LM"
)

# 4、获取peft_model
peft_model = get_peft_model(model,lora_config)

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1344.01it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
from trl.trainer.sft_config import SFTConfig
import os
os.environ["TENSORBOARD_LOGGING_DIR"]="./logs/06_lora_demo"

# 5、sftconfig实例
config = SFTConfig(
    per_device_train_batch_size=4,
    per_device_eval_batch_size= 4,
    gradient_accumulation_steps=8,
    max_steps=300,
    logging_strategy="steps",
    logging_steps=10,
    report_to="tensorboard",
    # 注意：LoRA微调的学习率，一般来说，会比全参微调，高一个数量级
    learning_rate=3e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.1,
    eval_strategy="steps",
    eval_steps=50,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
    optim = "paged_adamw_32bit", # 可选择性使用 分页优化器
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    output_dir="./finetuned/06_lora_demo",
    bf16=True,
    gradient_checkpointing=False,
    activation_offloading=False,
    max_length= 650,
    assistant_only_loss=True,
    chat_template_path="./new_chat_template.jinja"
)

In [ ]:
from trl.trainer.sft_trainer import SFTTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer

# 6、构造trainer
trainer = SFTTrainer(
    model=peft_model,
    args=config,
    processing_class=tokenizer,
    train_dataset=mapped_data["train"],
    eval_dataset=mapped_data["test"],
)

# 7、训练和保存
trainer.train()
# 保存模型参数，和Tokenizer相关的配置，从而使得，后面，可以通过加载这个路径，得到model和tokenizer
trainer.save_model("./finetuned/06_lora_demo")